In [53]:
import pyspark
from pyspark.sql import SparkSession

In [54]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

In [55]:
spark.sparkContext.uiWebUrl

'http://mac.lan:4041'

In [56]:
!wget -P ./data/ https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-01 15:18:05--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-01T21%3A56%3A20Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-01T20%3A55%3A21Z&ske=2026-03-01T21%3A56%3A20Z&sks=b&skv=2018-11-09&sig=No0OcyFT6zvEZJwlL8FdnijCdpfS90zXsKdjEqtwZSg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjQwMzQ4NSwibmJmIjoxNzcyMzk5ODg1LCJwYXRoIjoi

In [57]:
df_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("./data/fhvhv_tripdata_2021-01.csv.gz")

In [58]:
df_inferred.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [59]:
# save schema to json
schema_json = df_inferred.schema.json()
print(schema_json)

{"fields":[{"metadata":{},"name":"hvfhs_license_num","nullable":true,"type":"string"},{"metadata":{},"name":"dispatching_base_num","nullable":true,"type":"string"},{"metadata":{},"name":"pickup_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"dropoff_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"PULocationID","nullable":true,"type":"integer"},{"metadata":{},"name":"DOLocationID","nullable":true,"type":"integer"},{"metadata":{},"name":"SR_Flag","nullable":true,"type":"integer"}],"type":"struct"}


In [60]:
# save schema to file
import json
from pathlib import Path

schema_path = Path("./schemas/fhvhv_tripdata.schema.json")

# pretty-print for readability
schema_dict = json.loads(schema_json)

with schema_path.open("w") as f:
    json.dump(schema_dict, f, indent=2)

print(f"Schema saved to {schema_path.resolve()}")

Schema saved to /Users/mcopple/src/zoomcamps/src/de-homework-2026/module-6/examples/notebooks/schemas/fhvhv_tripdata.schema.json


In [61]:
# now try to read the data again, but this time with the schema
from pyspark.sql.types import StructType
with open("./schemas/fhvhv_tripdata.schema.json") as f:
    schema_dict = json.load(f)

schema = StructType.fromJson(schema_dict)

df = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("./data/fhvhv_tripdata_2021-01.csv.gz")

In [62]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', IntegerType(), True)])

In [63]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [64]:
# repartition dataframe to enable parallel processing
df = df.repartition(24)

In [65]:
df.write.parquet('./data/fhvhv/2021/01/')

26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/01 15:22:03 WARN MemoryManager: Total allocation exceeds 95.

In [67]:
df = spark.read.parquet('./data/fhvhv/2021/01/')

In [68]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: int]

In [69]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID').filter(df.hvfhs_license_num == 'HV0003').show(
    truncate=False
)

+-------------------+-------------------+------------+------------+
|pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 19:23:33|2021-01-02 19:31:20|223         |179         |
|2021-01-13 08:58:46|2021-01-13 09:02:49|181         |106         |
|2021-01-29 15:21:13|2021-01-29 15:38:18|226         |82          |
|2021-01-18 21:37:13|2021-01-18 21:48:38|61          |72          |
|2021-01-02 18:50:17|2021-01-02 19:04:44|17          |256         |
|2021-01-24 08:18:27|2021-01-24 08:31:15|196         |173         |
|2021-01-16 19:18:50|2021-01-16 19:27:42|23          |118         |
|2021-01-16 23:05:14|2021-01-16 23:12:57|61          |61          |
|2021-01-31 13:24:26|2021-01-31 13:36:07|249         |170         |
|2021-01-07 16:21:50|2021-01-07 16:33:42|130         |28          |
|2021-01-05 15:23:53|2021-01-05 15:50:56|138         |205         |
|2021-01-18 13:15:20|2021-01-18 13:45:55|141    